# Quick SLM — 02 · Pretraining

Pretrains the 103 M-parameter model on `combined.bin`. One GPU, resumable, ~9 390 steps
to reach the 10 B-token budget at 260 sequences x 4096 ctx per step.

Everything below calls into `quick_slm_trainer`. Two behaviours differ from the original
notebook and both are corrections, not tuning:

1. **The dataloader now shards across workers.** An `IterableDataset` is copied into every
   worker, and with an identical shuffle seed all four walked the same permutation, so the
   DataLoader handed back four consecutive duplicates of every batch. The v1 run therefore
   saw about a quarter of the corpus, four times each. `loader.shard` fixes it, and both stages' datasets go through it.
2. **Gradient accumulation is token-weighted.** For pretraining this is an identity, since
   every micro-batch scores the same number of tokens. It matters at SFT, and the trainer
   is shared, so the correct form is used in both.

The architecture, LR schedule, batch shape, and checkpoint cadence are `pretrain_v1()`,
which records what the v1 run actually did. `training/IMPORTANT_NOTES.md` argues the peak
LR should have been 6e-4 to 1e-3; `pretrain_v2()` holds that corrected recipe.

> **V1 EDUCATIONAL DEFECTS: Read before proceeding!**
> 
> This v1 notebook is preserved exactly as it ran to serve as an educational record of the SLM training process, including its flaws. The technical paper details three major defects present in this configuration:
> 
> 1. **Gradient Accumulation & Optimizer Steps**: `GRAD_ACCUM` was set to 13 to hit an arbitrary batch size, which accidentally bottlenecked the total optimizer steps to 9,390. This crippled the $\Sigma \eta$ schedule, meaning the learning rate didn't have enough steps to move the parameters effectively. (Choose step count first, derive accum second!)
> 2. **Learning Rate**: `LR_PEAK = 3e-4` was too low for a 1.06M token batch size. It should have been at least `6e-4` or `8e-4`. A peak LR must be matched to batch size, not parameter count.
> 3. **Context Length**: `ctx=4096` was heavily over-budgeted. Most FineWeb-Edu documents are under 2K tokens, meaning ~25% of the compute was wasted on padding and concatenating unrelated documents.



## 1 · Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

## 2 · Framework

`transformers`' import chain pulls in `torchvision`. If Colab's preinstalled `torchvision`
was built against a different CUDA version than `torch`, the import explodes the moment
`LlamaConfig` is touched. Pretraining needs neither, so they are removed.

In [ ]:
import os
# Let the allocator grow one segment instead of fragmenting many. Helps near OOM,
# costs nothing.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip uninstall -q -y torchvision torchaudio

In [ ]:
EXTRAS = "data"

# Framework and data live on Drive (no GitHub). Upload the repo folder
# (with pyproject.toml, README.md, and framework/) into DRIVE_ROOT/code once.
# Every notebook installs it from there.
DRIVE_ROOT = "/content/drive/MyDrive/quick-slm"

import subprocess, sys
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / "code", root, root / "quick-slm"]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    looked = "\n  ".join(str(p) for p in candidates)
    raise RuntimeError(
        "No pyproject.toml found on Drive. Upload the repo (pyproject.toml, "
        f"README.md, and framework/) to {root / 'code'}, then re-run.\nLooked in:\n  "
        + looked
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

import v1.quick_slm_trainer as q
print("quick-slm-trainer", q.__version__, "from", Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, "require_framework"):
    q.require_framework("v1", REPO_DIR)
else:
    raise RuntimeError(
        "quick-slm-trainer " + q.__version__ + " predates the support-window check; "
        "training v1 requires >=1.0. See SUPPORT.md."
    )


## 3 · Corpus

The manifest written by notebook 01 is the single source of truth for where `combined.bin`
lives, the CTX it was packed at, and the vocabulary it was tokenized with. It is mirrored
to local SSD so the random-window reads run at multi-GB/s rather than through Drive's mount.

In [ ]:
import json
from pathlib import Path

from v1.quick_slm_trainer import Config, Layout, pretrain_v1
from v1.quick_slm_trainer.pretraining import mirror_to_local
from v1.quick_slm_trainer.evaluate import corpus_windows
from v1.quick_slm_trainer.tokenizer import load_tokenizer

layout = Layout().mkdirs()
manifest = json.loads(layout.manifest_path.read_text())

CTX = int(manifest['ctx'])
VOCAB = int(manifest['vocab_size'])

cfg = pretrain_v1()
cfg.data.ctx = CTX

tok = load_tokenizer(layout.tokenizer_dir, patch=False)
assert len(tok) == VOCAB, f'tokenizer/manifest vocab mismatch: {len(tok)} vs {VOCAB}'

CORPUS = mirror_to_local(Path(manifest['combined_path']), layout.local_combined)

TOTAL_STEPS = cfg.run.total_steps(CTX)
print(f'corpus windows : {corpus_windows(CORPUS, CTX):,}')
print(f'tokens / step  : {cfg.run.tokens_per_step(CTX):,}')
print(f'total steps    : {TOTAL_STEPS:,}')

## 4 · Model, optimizer, resume

In [ ]:
from v1.quick_slm_trainer.model import build_model, enable_tf32, param_count, prepare, set_seed
from v1.quick_slm_trainer.optim import build_optimizer
from v1.quick_slm_trainer.trainer import Trainer, resume_if_possible

set_seed(cfg.run.seed)
enable_tf32()

model = build_model(cfg, vocab_size=VOCAB, tok=tok)
print(f'parameters : {param_count(model):,}')

model, fp8_active = prepare(model, cfg.run)
optimizer = build_optimizer(model, cfg.optim, device=cfg.run.device)
state = resume_if_possible(layout.ckpt_dir, model, optimizer)

## 5 · Loaders and evaluation

`make_loader` is handed to the trainer rather than a DataLoader, because the loop rebuilds
it when the corpus is exhausted. The eval slice is a fixed random draw from the same
corpus: at one epoch over 10 B tokens the distinction from a true held-out set is academic.

In [ ]:
import torch

from v1.quick_slm_trainer.loader import make_loader as build_loader
from v1.quick_slm_trainer.pretraining import WindowedMemmapDataset
from v1.quick_slm_trainer.evaluate import evaluate_pretrain

DTYPE = torch.bfloat16


def make_loader(start_window: int, seed: int):
    ds = WindowedMemmapDataset(CORPUS, CTX, start_window=start_window, seed=seed)
    return build_loader(
        ds,
        batch_size=cfg.run.micro_batch,
        num_workers=cfg.run.num_workers,
        prefetch_factor=cfg.run.prefetch_factor,
    )


def evaluate():
    return evaluate_pretrain(
        model, CORPUS, CTX,
        micro_batch=cfg.run.micro_batch, n_batches=cfg.run.eval_batches,
        device=cfg.run.device, dtype=DTYPE,
    )

## 6 · Train

A `tqdm` bar over the remaining steps, a line every 10 steps with smoothed loss, throughput,
grad-norm, MFU, and ETA, an eval every 500, and a checkpoint every 500. Metrics also land in
`logs/train.jsonl`, which `paper/scripts/plot_training.py` reads.

Reported MFU uses the 6N approximation and ignores attention, so it is a lower bound.

In [ ]:
trainer = Trainer(
    config=cfg,
    model=model,
    optimizer=optimizer,
    make_loader=make_loader,
    total_steps=TOTAL_STEPS,
    ctx=CTX,
    ckpt_dir=layout.ckpt_dir,
    log_path=layout.logs_dir / 'train.jsonl',
    tokenizer=tok,
    evaluate=evaluate,
    state=state,
    title='Quick SLM — pretraining (103M)',
    fp8_active=fp8_active,
)

state = trainer.train()

## 7 · Final artifact

Written in HuggingFace format so `from_pretrained` loads it directly.

In [ ]:
from v1.quick_slm_trainer.checkpoint import save_final

final = save_final(layout.final_dir, model, tokenizer=tok, config=cfg)
print('final model written to', final)
for f in sorted(final.iterdir()):
    print(f'  {f.name:<28s} {f.stat().st_size / 1e6:>9.2f} MB')